# 🚀 Autonomous Treasury Mesh — Llama 3.3 / 3.1 Inference Tunnel for Google Colab

This notebook runs an OpenAI-compatible server using **Ollama** and exposes a public endpoint via **Cloudflare Tunnel**.
You can plug the resulting URL directly into the `NEAR_AI_BASE_URL` in your `.env` file.

### Recommended Runtime:
- **Colab Free (T4 GPU)**: Run `llama3.1` (8B parameter model, ultra-fast and lightweight)
- **Colab Pro (A100 GPU)**: Run `llama3.3` (70B parameter model)

### Step 1: Install Ollama & Cloudflare Tunnel

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null
print("✔ Installed Ollama & Cloudflare Tunnel successfully.")

### Step 2: Start Ollama Background Server & Pull Model

In [ ]:
import subprocess, time

# Start Ollama server in background
subprocess.Popen(["ollama", "serve"])
time.sleep(3)

# Pull model (Change to llama3.3 if using A100 GPU)
MODEL_NAME = "llama3.1"
print(f"Pulling model: {MODEL_NAME}...")
!ollama pull {MODEL_NAME}
print(f"✔ Model {MODEL_NAME} is ready!")

### Step 3: Launch Cloudflare Tunnel and Get Public URL
Copy the URL generated below and paste it into your `web/.env` file as `NEAR_AI_BASE_URL`.

In [ ]:
import re

# Run tunnel in background
!cloudflared tunnel --url http://localhost:11434 --logfile tunnel.log &
time.sleep(5)

# Retrieve public URL
with open("tunnel.log", "r") as f:
    content = f.read()
    matches = re.findall(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', content)
    if matches:
        tunnel_url = matches[-1]
        print("\n=======================================================")
        print("🎉 Your OpenAI-compatible Colab Endpoint:")
        print(f"   URL: {tunnel_url}/v1")
        print("=======================================================\n")
        print("Paste these lines into your web/.env file:")
        print(f'NEAR_AI_BASE_URL="{tunnel_url}/v1"')
        print('NEAR_AI_API_KEY="colab-test"')
        print(f'NEAR_AI_MODEL="{MODEL_NAME}"')
    else:
        print("Tunnel is initializing, please rerun this cell in 5 seconds.")